# Reitz 2017 — Empirical Recharge Estimates

Visualize the consolidated Reitz et al. (2017) recharge dataset fetched
from USGS ScienceBase (doi:10.5066/F7PN93P0).

- **Variables:** `total_recharge`, `eff_recharge` (base flow component)
- **Resolution:** ~800m CONUS (geographic coordinates, WGS 84)
- **Period:** 2000–2013 (annual)
- **Units:** inches/year

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH = DATASTORE / "reitz2017" / "reitz2017_consolidated.nc"


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid: {ds.sizes.get('y', 'N/A')} y x {ds.sizes.get('x', 'N/A')} x")
print(f"Variables: {list(ds.data_vars)}")
for var in ds.data_vars:
    print(f"  {var}: dtype={ds[var].dtype}, shape={ds[var].shape}")


## Single-year recharge maps (2005)

In [ ]:
TARGET_YEAR = "2005-07-01"

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, var, title in zip(
    axes,
    ["total_recharge", "eff_recharge"],
    ["Total Recharge", "Effective Recharge"],
):
    da = ds[var].sel(time=TARGET_YEAR, method="nearest")
    da.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "inches/year"})
    ax.set_title(f"{title} — 2005")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

for var in ["total_recharge", "eff_recharge"]:
    da = ds[var].sel(time=TARGET_YEAR, method="nearest")
    print(f"\n{var} (2005):")
    print(f"  min:  {float(da.min()):.2f}")
    print(f"  max:  {float(da.max()):.2f}")
    print(f"  mean: {float(da.mean()):.2f}")
    print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## Mean recharge over calibration period (2000–2009)

In [ ]:
calib = ds.sel(time=slice("2000-01-01", "2009-12-31"))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, var, title in zip(
    axes,
    ["total_recharge", "eff_recharge"],
    ["Total Recharge", "Effective Recharge"],
):
    mean_da = calib[var].mean(dim="time")
    mean_da.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "inches/year"})
    ax.set_title(f"Mean {title} — 2000–2009")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

for var in ["total_recharge", "eff_recharge"]:
    mean_da = calib[var].mean(dim="time")
    print(f"\n{var} mean (2000-2009):")
    print(f"  min:  {float(mean_da.min()):.2f}")
    print(f"  max:  {float(mean_da.max()):.2f}")
    print(f"  mean: {float(mean_da.mean()):.2f}")


## Difference: Total vs. Effective recharge (2000–2009 mean)

In [ ]:
diff = calib["total_recharge"].mean(dim="time") - calib["eff_recharge"].mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 8))
diff.plot(ax=ax, cmap="RdBu_r", robust=True, cbar_kwargs={"label": "inches/year"})
ax.set_title("Total Recharge minus Effective Recharge — Mean 2000–2009")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Difference stats (total - effective):")
print(f"  min:  {float(diff.min()):.2f}")
print(f"  max:  {float(diff.max()):.2f}")
print(f"  mean: {float(diff.mean()):.2f}")


## Annual time series (CONUS spatial mean)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for var, label, color in [
    ("total_recharge", "Total Recharge", "#2980b9"),
    ("eff_recharge", "Effective Recharge", "#e67e22"),
]:
    annual_mean = ds[var].mean(dim=["y", "x"])
    ax.plot(ds.time.values, annual_mean.values, marker="o", label=label, color=color)

ax.axvspan("2000-01-01", "2009-12-31", alpha=0.12, color="green", label="Calibration period")
ax.set_ylabel("Mean Recharge (inches/year)")
ax.set_title("CONUS-mean annual recharge — 2000–2013")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Year-to-year variability (total recharge)

In [ ]:
n_years = ds.sizes["time"]
ncols = 4
nrows = (n_years + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))

for i, t in enumerate(ds.time.values):
    ax = axes.flatten()[i]
    year = str(t)[:4]
    ds["total_recharge"].sel(time=t).plot(
        ax=ax, cmap="YlGnBu", robust=True, add_colorbar=False
    )
    ax.set_title(year)
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")

# Hide unused axes
for j in range(i + 1, nrows * ncols):
    axes.flatten()[j].set_visible(False)

fig.suptitle("Total Recharge by Year (inches/year)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
ds


## CRS and spatial metadata

In [ ]:
import rioxarray  # noqa: F401

da = ds["total_recharge"].isel(time=0)
print("Spatial metadata:")
print(f"  Dimensions: {dict(da.sizes)}")
print(f"  y range:    {float(da.y.min()):.4f} to {float(da.y.max()):.4f}")
print(f"  x range:    {float(da.x.min()):.4f} to {float(da.x.max()):.4f}")

# Read CRS from CF grid_mapping variable
if "crs" in ds:
    crs_attrs = ds["crs"].attrs
    print(f"  CRS:        {crs_attrs.get('crs_wkt', 'N/A')[:60]}...")
    print(f"  Grid map:   {crs_attrs.get('grid_mapping_name', 'N/A')}")
elif da.rio.crs is not None:
    print(f"  CRS:        {da.rio.crs}")
    print(f"  EPSG:       {da.rio.crs.to_epsg()}")
else:
    print("  CRS:        not set (re-run consolidation to embed CRS)")

dy = abs(float(da.y[1] - da.y[0]))
dx = abs(float(da.x[1] - da.x[0]))
print(f"  Resolution: {dy:.6f}° x {dx:.6f}°  (~{dy * 111_000:.0f}m x {dx * 111_000 * np.cos(np.radians(37)):.0f}m at 37°N)")
print(f"  Conventions: {ds.attrs.get('Conventions', 'N/A')}")


## Manifest provenance check

In [ ]:
import json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    entry = manifest["sources"].get("reitz2017", {})
    print(f"Source key:     {entry.get('source_key', 'N/A')}")
    print(f"DOI:            {entry.get('doi', 'N/A')}")
    print(f"License:        {entry.get('license', 'N/A')}")
    print(f"Period:         {entry.get('period', 'N/A')}")
    print(f"Spatial extent: {entry.get('spatial_extent', 'N/A')}")
    print(f"Variables:      {entry.get('variables', 'N/A')}")
    print(f"File path:      {entry.get('file', {}).get('path', 'N/A')}")
    print(f"Size (bytes):   {entry.get('file', {}).get('size_bytes', 'N/A')}")
    print(f"Downloaded:     {entry.get('file', {}).get('downloaded_utc', 'N/A')}")
    print(f"Years:          {entry.get('file', {}).get('years', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")
